# Ambil Dataset Sentimen dari Hugging Face

Hugging Face punya ribuan dataset NLP yang siap pakai, termasuk sentiment analysis bahasa Indonesia.

**Kelebihan:**
- Tidak perlu API key
- Dataset sudah dilabeli
- Ada yang bahasa Indonesia
- Download langsung pakai library `datasets`

In [ ]:
!pip install datasets pandas

In [ ]:
from datasets import load_dataset
import pandas as pd

## 1. IndoNLU SMSA - Sentiment Analysis Bahasa Indonesia

Dataset review film dalam bahasa Indonesia dengan 3 label:
- **positive** (positif)
- **negative** (negatif)
- **neutral** (netral)

Sumber: https://huggingface.co/datasets/indonlu

In [ ]:
# Load dataset IndoNLU SMSA
smsa = load_dataset('indonlu', 'smsa')

print('Struktur dataset:')
print(smsa)
print(f'\nKolom: {smsa["train"].column_names}')
print(f'Jumlah data training: {len(smsa["train"])}')
print(f'Jumlah data validation: {len(smsa["validation"])}')
print(f'Jumlah data test: {len(smsa["test"])}')

In [ ]:
# Convert ke pandas DataFrame
df_train = smsa['train'].to_pandas()
df_test = smsa['test'].to_pandas()

# Gabung semua data
df = pd.concat([df_train, df_test], ignore_index=True)

print(f'Total data: {len(df)}')
print(f'\nDistribusi label:')
print(df['label'].value_counts())
print()
df.head(10)

In [ ]:
# Mapping label angka ke teks
label_map = {0: 'negatif', 1: 'netral', 2: 'positif'}
df['sentimen'] = df['label'].map(label_map)

# Cek contoh tiap sentimen
print('=== CONTOH POSITIF ===')
print(df[df['label'] == 2]['text'].iloc[0])
print()
print('=== CONTOH NEGATIF ===')
print(df[df['label'] == 0]['text'].iloc[0])
print()
print('=== CONTOH NETRAL ===')
print(df[df['label'] == 1]['text'].iloc[0])

## 2. Wongnai Reviews - Sentiment Analysis (Bahasa Thailand, untuk perbandingan)

Kalau mau coba dataset bahasa lain juga bisa.

In [ ]:
# Contoh dataset bahasa lain
imdb = load_dataset('imdb', split='train[:100]')  # ambil 100 data saja untuk demo

df_imdb = imdb.to_pandas()
print(f'Dataset IMDB (review film Inggris):')
print(f'Jumlah: {len(df_imdb)}')
print(f'Label: {df_imdb["label"].value_counts().to_dict()}')
df_imdb.head()

## 3. Cari Dataset Lain di Hugging Face

Ada ribuan dataset di Hugging Face. Berikut cara cari yang relevan.

In [ ]:
from huggingface_hub import HfApi

api = HfApi()

# Cari dataset dengan kata kunci "sentiment indonesian"
datasets = api.list_datasets(search='sentiment indonesian', limit=10)

print('Dataset sentiment bahasa Indonesia yang tersedia:\n')
for ds in datasets:
    print(f'  - {ds.id}')

## 4. Pakai Dataset IndoNLU untuk Training Model

Sekarang kita pakai dataset SMSA untuk latih model sentiment analysis.

In [ ]:
import re
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import classification_report, accuracy_score

# Preprocessing sederhana
def bersihkan(teks):
    teks = teks.lower()
    teks = re.sub(r'[^a-z\s]', '', teks)
    teks = re.sub(r'\s+', ' ', teks).strip()
    return teks

df['text_bersih'] = df['text'].apply(bersihkan)

# TF-IDF
tfidf = TfidfVectorizer(max_features=5000)
X = tfidf.fit_transform(df['text_bersih'])
y = df['label']

# Split (pakai data dari IndoNLU yang sudah di-split)
df_train_hf = smsa['train'].to_pandas()
df_test_hf = smsa['test'].to_pandas()

X_train = tfidf.fit_transform(df_train_hf['text'].apply(bersihkan))
X_test = tfidf.transform(df_test_hf['text'].apply(bersihkan))
y_train = df_train_hf['label']
y_test = df_test_hf['label']

# Training
model = MultinomialNB()
model.fit(X_train, y_train)

# Evaluasi
y_pred = model.predict(X_test)
print(f'Akurasi: {accuracy_score(y_test, y_pred):.2%}')
print(f'\nClassification Report:')
print(classification_report(y_test, y_pred, target_names=['negatif', 'netral', 'positif']))

## 5. Uji dengan Teks Baru

In [ ]:
def prediksi(teks):
    teks_bersih = bersihkan(teks)
    X = tfidf.transform([teks_bersih])
    label = model.predict(X)[0]
    label_map = {0: 'NEGATIF', 1: 'NETRAL', 2: 'POSITIF'}
    print(f'"{teks}" => {label_map[label]}')

prediksi('Film ini sangat bagus dan menarik')
prediksi('Ceritanya membosankan dan jelek')
prediksi('Film biasa saja tidak ada yang spesial')
prediksi('Aktingnya keren sekali recommended')

## 6. Simpan Dataset ke CSV

Simpan untuk dipakai di notebook lain.

In [ ]:
df.to_csv('huggingface_sentiment_dataset.csv', index=False)
print(f'Dataset tersimpan: huggingface_sentiment_dataset.csv')
print(f'Total: {len(df)} baris')

## Ringkasan

| Dataset | Bahasa | Ukuran | Label |
|---|---|---|---|
| IndoNLU SMSA | Indonesia | ~11.000 | positif, netral, negatif |
| IMDB | Inggris | 50.000 | positif, negatif |
| Wongnai | Thailand | 40.000+ | 1-5 bintang |

**Tips:**
- Browse dataset lain di https://huggingface.co/datasets
- Filter by task: "text-classification" + language: "indonesian"
- Gunakan `load_dataset('nama_dataset')` untuk load